# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Softwareguy786/ML_INTERNSHIP_01/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: the paper reports Random Forest NDCG@10 = 0.690 vs a rule baseline of 0.066 on 180 held-out observations. My methodology question: where does the label come from? The label is next-day GSC clicks — a future proxy, not an observed refresh need. The time-aware validation is a strength, but the window covers only five dates and two clients, so the claim is bounded to that slice.

Finding 2: 171 of 180 evaluation targets (95%) are zero. My methodology question: does that validation design carry the claim? With almost every target at zero, NDCG@10 is driven by very few non-zero rows, so the gap may be sensitive to a handful of items. The paper handles this honestly by calling the result "improved ordering," not prediction — which is the right level of claim.

In [6]:
import pandas as pd

df = pd.read_csv('/content/content_refresh_anonymized.csv')

print("shape:", df.shape)
print("columns:", df.columns.tolist())
for c in ['client_id', 'site_id', 'date', 'snapshot_date', 'content_id', 'page_id']:
    if c in df.columns:
        print(c, "->", df[c].nunique(), "unique")

shape: (30000, 44)
columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
client_id -> 32 unique
content_id -> 30000 unique


## 2. My model under an honest split (before/after)

*Under a random split, the model scored F1 = 0.680. Under GroupKFold by client_id — where no client appears in both train and test — average F1 was 0.661, only slightly lower. But the per-fold scores ranged from 0.457 to 0.799. That spread is the real result: the model's performance is unstable across client groups. It works well for some clients and poorly for others. The random-split number (0.680) was not badly inflated, but it hid this instability. Any recommendation drawn from this model is directional and decision-support, not a fixed prediction.*

In [7]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score

df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

features = ['content_age_days','days_since_last_update','impressions_90d',
            'clicks_90d','word_count','ctr']

X = df[features].fillna(0)
y = df['is_declining']
groups = df['client_id']

# --- BEFORE: random split (same as w05) ---
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
m_rand = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
p_rand = m_rand.predict(Xte)

# --- AFTER: grouped by client_id ---
gkf = GroupKFold(n_splits=5)
f1s, ps, rs = [], [], []
for tr_idx, te_idx in gkf.split(X, y, groups):
    m = LogisticRegression(max_iter=1000).fit(X.iloc[tr_idx], y.iloc[tr_idx])
    p = m.predict(X.iloc[te_idx])
    f1s.append(f1_score(y.iloc[te_idx], p))
    ps.append(precision_score(y.iloc[te_idx], p))
    rs.append(recall_score(y.iloc[te_idx], p))

# --- Compare ---
compare = pd.DataFrame({
    'Split': ['Random 80/20 (w05)', 'GroupKFold by client_id (w06)'],
    'F1':    [f1_score(yte, p_rand), np.mean(f1s)],
    'Precision': [precision_score(yte, p_rand), np.mean(ps)],
    'Recall':    [recall_score(yte, p_rand), np.mean(rs)],
}).round(3)

print(compare.to_string(index=False))
print("\nGroupKFold F1 per fold:", [round(v,3) for v in f1s])

                        Split    F1  Precision  Recall
           Random 80/20 (w05) 0.680      0.611   0.767
GroupKFold by client_id (w06) 0.661      0.579   0.775

GroupKFold F1 per fold: [0.587, 0.773, 0.457, 0.799, 0.689]


## 3. Leakage audit

*Two columns in the dataset derive from the target: trend_direction and trend_pct. Neither is in my feature list — overlap is an empty set. I also checked ctr, since it could in principle be computed from a window overlapping the trend window. Its correlation with trend_pct is 0.008, essentially zero, so it is not a disguised trend signal. My six features are clean: they describe the current snapshot only, not the change that defines the label. One remaining risk I cannot rule out: if the 90-day windows used for impressions and clicks overlap the 30-day windows that feed trend_direction, some indirect leakage may exist. The data contract does not state the window boundaries explicitly, so I flag this as an open question rather than claim it is resolved.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*My boldest Week-5 claim was: "Logistic regression reaches F1 = 0.68, a real improvement over the baseline." Rewritten in safe language: "Under a random split, logistic regression measured F1 = 0.68 against a rule baseline of 0.009. Under a client-grouped split the average was 0.661 with a per-fold range of 0.457 to 0.799. The model's ordering is directionally better than the rule, but performance varies by client group, so the output is decision-support for a human reviewer — not a prediction of which pages will decline."*

In [9]:
# Show the boldest vs rewritten claim side by side
claims = pd.DataFrame({
    'Version': ['Week 5 (bold)', 'Week 6 (safe)'],
    'Claim': [
        'F1 = 0.68, a real improvement.',
        'F1 = 0.68 random / 0.661 grouped, range 0.457-0.799. Directionally better; decision-support.'
    ],
    'Words used': ['improvement', 'measured, directional, decision-support']
})
print(claims.to_string(index=False))

      Version                                                                                        Claim                              Words used
Week 5 (bold)                                                               F1 = 0.68, a real improvement.                             improvement
Week 6 (safe) F1 = 0.68 random / 0.661 grouped, range 0.457-0.799. Directionally better; decision-support. measured, directional, decision-support


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.